## ▶ What you'll see when you run this
- A POST to `/ask` with *"12 + 30, and the price of coffee?"* returning a JSON answer that **combines both tools** — `add=42, get_price=4` — served by the same agent code as `agent_app.py`.

**Time:** ~8 min · **Cost:** free (in-process Flask test client) · **Keys:** none required (reuses the fallback agent) — set `ANTHROPIC_API_KEY` to serve the real Claude loop.

# Week 13 · Notebook 2 — Serving an Agent with Flask
**CSCI 250 · Fall 2026**

Put the agent loop behind an HTTP endpoint so others can use it. The same code is in `code/agent_app.py` (run it as a real script). 

> Colab can't easily hold a long-running server, so here we **test the Flask app in-process** with its test client — no network, no key needed.

## 1. The app factory
`run_agent` is pluggable. We **reuse the real agent from `agent_app.py`** — `run_claude_agent` (when `ANTHROPIC_API_KEY` is set) or `run_fallback_agent` (no key) — both of which already handle **add + get_price**. So the documented curl (`12 + 30, and the price of coffee?`) works end-to-end here too.

In [ ]:
import os, sys
from flask import Flask, request, jsonify

# Make agent_app.py importable: it sits next to this notebook in code/.
for p in ('.', 'code', '../code', '../../code'):
    if os.path.exists(os.path.join(p, 'agent_app.py')):
        sys.path.insert(0, os.path.abspath(p)); break

try:
    from agent_app import run_claude_agent, run_fallback_agent
    print('Reusing run_claude_agent / run_fallback_agent from agent_app.py')
except Exception as e:
    # Inline fallback so the notebook still runs if agent_app.py isn't on the path.
    print('agent_app.py not found, using inline fallback:', e)
    import re
    _TOOLS = {'add': lambda a, b: a + b,
              'get_price': lambda item: {'coffee': 4, 'tea': 3, 'cocoa': 5}
                          .get(str(item).lower(), 0)}
    def run_fallback_agent(user_msg, max_turns=5):
        calls = []
        m = re.search(r'(\d+)\s*\+\s*(\d+)', user_msg)
        if m: calls.append(('add', (int(m.group(1)), int(m.group(2)))))
        for item in ('coffee', 'tea', 'cocoa'):
            if item in user_msg.lower(): calls.append(('get_price', (item,)))
        obs = [f'{n}={_TOOLS[n](*a)}' for n, a in calls[:max_turns]]
        return '(fallback) ' + (', '.join(obs) if obs else 'no tool matched')
    run_claude_agent = run_fallback_agent

def make_agent():
    """Return a run_agent(message)->str: real Claude loop or fallback."""
    return run_claude_agent if os.environ.get('ANTHROPIC_API_KEY') else run_fallback_agent

def create_app():
    app = Flask(__name__)
    run_agent = make_agent()
    @app.route('/ask', methods=['POST'])
    def ask():
        msg = request.get_json(force=True).get('message', '')
        return jsonify({'answer': run_agent(msg)})
    @app.route('/health')
    def health():
        return jsonify({'ok': True})
    return app

## 2. Test it in-process (no server, no key)
Flask's test client lets us POST to `/ask` without starting a real server. The request mirrors the documented curl, so the answer uses **both** tools.

In [ ]:
app = create_app()
client = app.test_client()
print('health:', client.get('/health').get_json())
r = client.post('/ask',
                json={'message': 'What is 12 + 30, and the price of coffee?'})
print('ask:', r.get_json())

## 3. Run it for real (outside Colab)
Locally, `python agent_app.py` then in another terminal:
```bash
curl -X POST localhost:5000/ask -H 'Content-Type: application/json' \
     -d '{"message": "What is 12 + 30, and the price of coffee?"}'
```
**Never hard-code your API key in the app** — it reads it from the environment. For A10, wire `run_agent` to your real tool-using loop from Notebook 1.

In [ ]:
# scratch:
